# AI Data-Quality Triage — a working demo

**The claim this notebook demonstrates:** *AI agents can take a raw data-quality alert and turn it into a complete, trustworthy incident diagnosis — automatically, in seconds, with humans involved only where risk demands it.*

The story: a small online shop (mugs, t-shirts, hoodies…). Something breaks in its sales data. Watch a team of small AI agents figure out **what** broke, **why**, **how much it matters**, and **what to do about it**.

What each part proves:

1. **Detection can stay dumb** — a simple threshold check fires the alert; a chart shows you the anomaly before any AI runs.
2. **The agents genuinely reason** — Act 2 reruns the *identical* agents on a different failure and gets a different, correct diagnosis.
3. **AI output can be trusted enough to act on** — Pydantic validates structure *and* meaning ("a confident root cause must cite evidence").
4. **Risk is governable** — P1 / high-risk / low-confidence outcomes route to a human; the rest may auto-apply.
5. **Waste is preventable** — a dedup gate short-circuits repeat alerts before any agent spends money.

| Layer | In this notebook | In production |
|---|---|---|
| Warehouse | in-memory SQLite | your real warehouse |
| Lineage + catalog | Python dicts | Glue / Collibra / dbt / OpenLineage via MCP |
| Detection | real checks (run here) | Great Expectations + ML drift |
| **AI agents** | **real** | **real (same code)** |
| **Pydantic validation** | **real** | **real** |
| **Orchestrator** | **real** | **real** |

> Runs out of the box in **MOCK mode** — no API key needed. Add an OpenAI or Anthropic key to a `.env` file to watch the agents reason live.
>
> *Honest boundary:* the enterprise plumbing is stubbed, so this demo confirms the **architecture**, not enterprise scale — and in production the diagnosis is only as good as your real lineage source.

## Setup

One install cell. `openai` / `anthropic` are only exercised in live mode — MOCK mode needs neither at runtime.

In [ ]:
%pip install -q pydantic anthropic openai python-dotenv pandas matplotlib

## Pick a mode

MOCK is the default: the full pipeline runs instantly with canned (but Pydantic-validated) agent responses. To go live, put `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` (and optionally `MODEL`) in a `.env` file next to this notebook's repo root. Anthropic wins if both are set. There is no silent fallback: live mode errors surface as errors.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("ANTHROPIC_API_KEY"):
    PROVIDER, MODEL = "anthropic", os.getenv("MODEL", "claude-sonnet-4-6")
elif os.getenv("OPENAI_API_KEY"):
    PROVIDER, MODEL = "openai", os.getenv("MODEL", "gpt-4o-mini")
else:
    PROVIDER, MODEL = "mock", None

print("MOCK mode — no LLM calls, canned responses" if PROVIDER == "mock"
      else f"LIVE mode — {PROVIDER} -> {MODEL}")